In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit


class SeismicLawValidator:

    def __init__(self, cleaned_csv: str):
        self.df = pd.read_csv(cleaned_csv)
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"], format="ISO8601", utc=True)

    # =========================================================
    # LAW 1: GUTENBERG-RICHTER (b-value)
    # =========================================================
    def validate_gutenberg_richter(
        self, mc: float = 2.5, max_fit_mag: float = 6.0
    ):
        print("=" * 50)
        print(" LAW 1: GUTENBERG-RICHTER MAGNITUDE-FREQUENCY")
        print("=" * 50)

        # Filter for complete catalog above Mc
        mags = self.df[self.df["magnitude"] >= mc]["magnitude"]

        # Calculate cumulative frequency N(>M)
        sorted_mags = np.sort(mags)
        unique_mags, counts = np.unique(sorted_mags, return_counts=True)
        cum_counts = np.cumsum(counts[::-1])[::-1]

        log_N = np.log10(cum_counts)

        # To get a stable b-value, seismologists fit the linear slope
        # strictly in the non-saturated zone (e.g., M2.5 to M6.0)
        fit_mask = (unique_mags >= mc) & (unique_mags <= max_fit_mag)
        x_fit = unique_mags[fit_mask]
        y_fit = log_N[fit_mask]

        # Linear Regression: y = a - b*x  --> (slope is -b)
        res = stats.linregress(x_fit, y_fit)

        b_value = -res.slope
        a_value = res.intercept
        b_error = res.stderr
        r_squared = res.rvalue**2

        print(f" -> b-value         = {b_value:.4f} ± {b_error:.4f}")
        print(f" -> a-value (log a) = {a_value:.4f}")
        print(f" -> R² Fit Score    = {r_squared:.4f}")

        if r_squared >= 0.95:
            print(" [PASSED] Gutenberg-Richter R² >= 0.95 benchmark met!")
        else:
            print(" [WARNING] Fit score fell below 0.95 threshold.")

        # --- Plotting ---
        plt.figure(figsize=(8, 5))
        plt.scatter(
            unique_mags,
            log_N,
            color="black",
            alpha=0.6,
            label="Observed Catalog",
        )
        plt.plot(
            x_fit,
            a_value - b_value * x_fit,
            color="red",
            linewidth=2,
            label=f"Fit (b={b_value:.2f}, R²={r_squared:.3f})",
        )
        plt.title("Gutenberg-Richter Law: Southern California (1996-2026)")
        plt.xlabel("Magnitude ($M$)")
        plt.ylabel("$\log_{10}$ Cumulative Number of Quakes $N(>M)$")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.savefig("gutenberg_richter_fit.png", dpi=300)
        plt.close()

    # ==============================

In [5]:
if __name__ == "__main__":
    validator = SeismicLawValidator("southern_california_cleaned_declustered.csv")
    validator.validate_gutenberg_richter(mc=2.5)

 LAW 1: GUTENBERG-RICHTER MAGNITUDE-FREQUENCY
 -> b-value         = 1.0788 ± 0.0031
 -> a-value (log a) = 7.0645
 -> R² Fit Score    = 0.9976
 [PASSED] Gutenberg-Richter R² >= 0.95 benchmark met!
